# Imports

In [39]:
import sys
import os
import joblib
from pathlib import Path

In [2]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(),"..","..")))

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [5]:
from sklearn.metrics import mean_squared_error

In [6]:
from multiple_time_series_cv import MultipleTimeSeriesCV

In [7]:
from train_model import TrainModel

In [8]:
data_store=Path(os.getcwd()).parent.parent/"Prepared_Data_Store"

# Loading Training Data

In [9]:
train_X=pd.read_csv(
    os.path.join(data_store,"train_X.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

In [10]:
train_X

open      high       low     close    volume  \
date       ticker                                                     
2010-02-22 A      -1.571355 -1.621298 -1.566455 -1.588083 -0.156541   
           ACGL   -1.576142 -1.574122 -1.564216 -1.568240  3.054903   
           ACN    -1.503369 -1.518475 -1.506436 -1.519097 -0.338131   
           ADI    -1.462160 -1.479389 -1.463275 -1.469124  0.904611   
           ADM    -0.916182 -0.923291 -0.902353 -0.917975 -0.511968   
...                     ...       ...       ...       ...       ...   
2016-02-23 XEL     2.407518  2.411885  2.433889  2.451646  0.019789   
           XOM     0.441488  0.424918  0.396251  0.373889 -0.689229   
           YUM     0.704938  0.711610  0.717225  0.707216 -0.155420   
           ZBH     0.796296  0.791065  0.747731  0.740059 -0.062927   
           ZBRA    0.685893  0.706261  0.695372  0.720551  0.941935   

                   dollar_volume  dollar_volume_7d  dollar_volume_15d  \
date       ticker                                                       
2010-02-22 A           -0.693067         -0.279116          -0.341175   
           ACGL         1.343749          1.911222           1.708494   
           ACN         -0.758531         -0.957332          -0.950981   
           ADI         -0.044209          1.348163           0.612157   
           ADM         -0.916221         -0.615638           0.118269   
...                          ...               ...                ...   
2016-02-23 XEL          0.918381          1.835798           3.370234   
           XOM         -0.714623         -0.035235           0.951681   
           YUM          0.092064          0.458539           1.878205   
           ZBH          0.442415         -0.049787           1.221948   
           ZBRA         0.907915          1.284902           1.452440   

                   dollar_volume_21d  dollar_volume_rank  ...  max_price_7d  \
date       ticker                                         ...                 
2010-02-22 A               -0.708899            0.014026  ...     -1.698506   
           ACGL             1.505118           -2.431023  ...     -1.574045   
           ACN             -0.963062            1.277944  ...     -1.529100   
           ADI              0.524131           -1.456467  ...     -1.513738   
           ADM              0.143165           -0.951449  ...     -0.967399   
...                              ...                 ...  ...           ...   
2016-02-23 XEL              3.235384           -1.767585  ...      2.360262   
           XOM              0.907162           -0.382588  ...      0.385850   
           YUM              1.685877           -0.314443  ...      0.643528   
           ZBH              1.633233            0.406150  ...      0.756699   
           ZBRA             1.484118           -0.243752  ...      0.644209   

                   max_price_15d  max_price_21d       rsi  bb_lower  bb_upper  \
date       ticker                                                               
2010-02-22 A           -1.765908      -1.805616  1.054291  0.995363 -0.970205   
           ACGL        -1.577768      -1.577905  0.378266  0.452331 -0.631467   
           ACN         -1.536885      -1.526227 -1.015514 -0.732887 -0.034553   
           ADI         -1.533649      -1.541487  0.919647  2.052763 -1.320108   
           ADM         -0.862954      -0.888895 -0.538155 -0.582187 -0.188531   
...                          ...            ...       ...       ...       ...   
2016-02-23 XEL          2.308206       2.281543  1.256485  1.384522 -0.184640   
           XOM          0.329605       0.297865  0.508361  1.772654  0.191105   
           YUM          0.618603       0.631358  0.117181  0.929545  0.083859   
           ZBH          0.837428       0.981501 -1.029809 -0.074962  1.317450   
           ZBRA         0.592091       0.563529  0.891800  3.425449 -0.673004   

                   avg_true_range      macd  macd_hist  macd_signal

In [11]:
train_Y=pd.read_csv(
    os.path.join(data_store,"train_Y.csv"),
    parse_dates=["date"],
    index_col=["date","ticker"]
)

In [12]:
train_Y

forward_returns_1d  forward_returns_5d  forward_returns_7d  \
date       ticker                                                               
2010-02-22 A                -0.009631            0.020025            0.002472   
           ACGL             -0.000275            0.013788           -0.005998   
           ACN              -0.010811            0.003002            0.000985   
           ADI              -0.024757            0.019836           -0.007765   
           ADM              -0.006374            0.010218           -0.002345   
...                               ...                 ...                 ...   
2016-02-23 XEL               0.002009           -0.002023            0.000253   
           XOM               0.003570            0.014099           -0.003628   
           YUM              -0.000420            0.051332            0.012996   
           ZBH               0.008444            0.005991            0.014881   
           ZBRA              0.017136            0.017967            0.015730   

                   forward_returns_15d  forward_returns_21d  
date       ticker                                            
2010-02-22 A                 -0.003291             0.011566  
           ACGL              -0.002398             0.001330  
           ACN                0.009022            -0.000707  
           ADI               -0.002070             0.011972  
           ADM               -0.001745            -0.000341  
...                                ...                  ...  
2016-02-23 XEL                0.001979             0.008585  
           XOM                0.004975            -0.004398  
           YUM                0.008769             0.019878  
           ZBH               -0.007044             0.000192  
           ZBRA              -0.036609            -0.018077  

[565488 rows x 5 columns]

# Model Training

## Model Params

In [13]:
ridge_alphas=np.logspace(-4,4,9)
ridge_alphas=sorted(list(ridge_alphas)+list(ridge_alphas*5))

In [14]:
train_y=train_Y["forward_returns_1d"]

In [15]:
training_period=63
test_period=10
n_splits=int(6*252/test_period)
lookahead=1

cv=MultipleTimeSeriesCV(
    training_period=training_period,
    test_period=test_period,
    n_splits=n_splits,
    lookahead=lookahead
)

In [16]:
ridge_scores=[]

## Training Various Ridge Models

In [17]:
for alpha in ridge_alphas:
    print(f"Training Ridge Regression with alpha: {alpha}")
    model=Ridge(alpha=alpha,fit_intercept=False,random_state=42)


    for train_idx,val_idx in cv.split(X=train_X):
        data_trainX=train_X.iloc[train_idx]
        data_trainY=train_y.iloc[train_idx]

        data_valX=train_X.iloc[val_idx]
        data_valY=train_y.iloc[val_idx]

        model.fit(X=data_trainX,y=data_trainY)

        val_preds=model.predict(data_valX)
        
        preds=data_valY.to_frame("yreal").assign(ypreds=val_preds)
        preds_by_day=preds.groupby("date")

        ic_day=preds_by_day.apply(lambda x: spearmanr(x.yreal,x.ypreds)[0]*100).to_frame("ic")
        rmse_day=preds_by_day.apply(lambda x: np.sqrt(mean_squared_error(y_true=x.yreal,y_pred=x.ypreds))).to_frame("rmse")

        score=pd.concat([ic_day,rmse_day],axis=1)

        ridge_scores.append(score.assign(ridge_alpha=alpha))        

Training Ridge Regression with alpha: 0.0001
Training Ridge Regression with alpha: 0.0005
Training Ridge Regression with alpha: 0.001
Training Ridge Regression with alpha: 0.005
Training Ridge Regression with alpha: 0.01
Training Ridge Regression with alpha: 0.05
Training Ridge Regression with alpha: 0.1
Training Ridge Regression with alpha: 0.5
Training Ridge Regression with alpha: 1.0
Training Ridge Regression with alpha: 5.0
Training Ridge Regression with alpha: 10.0
Training Ridge Regression with alpha: 50.0
Training Ridge Regression with alpha: 100.0
Training Ridge Regression with alpha: 500.0
Training Ridge Regression with alpha: 1000.0
Training Ridge Regression with alpha: 5000.0
Training Ridge Regression with alpha: 10000.0
Training Ridge Regression with alpha: 50000.0


In [18]:
ridge_scores=pd.concat(ridge_scores,axis=0)

# Evaluating Ridge Models

In [19]:
ridge_scores.groupby("ridge_alpha").ic.describe()

,count,mean,std,min,25%,50%,75%,max
ridge_alpha,,,,,,,,
0.0001,1450.0,1.276460,10.985384,-38.174372,-5.572958,1.317571,8.322456,42.750091
0.0005,1450.0,1.276628,10.985207,-38.164967,-5.573740,1.321253,8.324478,42.744769
0.0010,1450.0,1.276617,10.985257,-38.164577,-5.571770,1.322929,8.325774,42.745274
0.0050,1450.0,1.276353,10.985982,-38.151135,-5.561148,1.323418,8.304752,42.762661
0.0100,1450.0,1.276891,10.986458,-38.155654,-5.571545,1.332347,8.293824,42.769864
0.0500,1450.0,1.278698,10.990598,-38.140767,-5.581431,1.379570,8.320906,42.861390
0.1000,1450.0,1.282223,10.994790,-38.171436,-5.606253,1.335356,8.259737,42.933716
0.5000,1450.0,1.296460,11.021108,-38.108354,-5.633395,1.449295,8.075785,42.944543
1.0000,1450.0,1.299767,11.042280,-37.874195,-5.775412,1.393484,8.210007,42.764864


In [20]:
ridge_scores.groupby("ridge_alpha").rmse.describe()

,count,mean,std,min,25%,50%,75%,max
ridge_alpha,,,,,,,,
0.0001,1450.0,0.016861,0.007134,0.006403,0.012533,0.015189,0.018987,0.081985
0.0005,1450.0,0.016861,0.007134,0.006403,0.012533,0.015189,0.018987,0.081985
0.0010,1450.0,0.016861,0.007133,0.006403,0.012533,0.015189,0.018987,0.081985
0.0050,1450.0,0.016861,0.007133,0.006403,0.012533,0.015188,0.018987,0.081984
0.0100,1450.0,0.016860,0.007133,0.006403,0.012533,0.015187,0.018987,0.081983
0.0500,1450.0,0.016858,0.007132,0.006403,0.012533,0.015183,0.018985,0.081978
0.1000,1450.0,0.016855,0.007130,0.006403,0.012533,0.015181,0.018983,0.081975
0.5000,1450.0,0.016840,0.007120,0.006401,0.012529,0.015170,0.018972,0.081975
1.0000,1450.0,0.016829,0.007113,0.006399,0.012520,0.015159,0.018970,0.081983


# Selecting Top-5 Ridge Models

In [21]:
score=ridge_scores.reset_index().set_index(["date","ridge_alpha"]).sort_index()

## Ranking Based on Daily IC

In [22]:
score["ic_rank"]=(score
                  .groupby("date")
                  .ic
                  .transform(lambda x: x.rank(ascending=False))
                 )

## Ranking Based on Daily RMSE

In [23]:
score["rmse_rank"]=(score
                    .groupby("date")
                    .rmse
                    .transform(lambda x: x.rank(ascending=True))
                   )

## Final Rank

In [24]:
score["final_rank"]=score[["ic_rank","rmse_rank"]].mean(axis=1)

In [25]:
score

ic      rmse  ic_rank  rmse_rank  final_rank
date       ridge_alpha                                                    
2010-05-20 0.0001      -1.693177  0.022876      2.0        1.0         1.5
           0.0005      -1.691159  0.022877      1.0        2.0         1.5
           0.0010      -1.696136  0.022877      3.0        3.0         3.0
           0.0050      -1.733618  0.022878      4.0        4.0         4.0
           0.0100      -1.746785  0.022879      5.0        5.0         5.0
...                          ...       ...      ...        ...         ...
2016-02-23 500.0000     3.384967  0.015559      7.0        5.0         6.0
           1000.0000    3.609171  0.015379      4.0        4.0         4.0
           5000.0000    5.507856  0.015143      3.0        1.0         2.0
           10000.0000   6.264009  0.015161      2.0        2.0         2.0
           50000.0000   7.344401  0.015361      1.0        3.0         2.0

[26100 rows x 5 columns]

## Model Comparison

In [26]:
model_comparison=(score
                  .swaplevel()
                  .sort_index()
                  .groupby("ridge_alpha")
                  .apply(lambda x: x.mean(axis=0))
                  .reset_index()
                 )

In [27]:
model_comparison

,ridge_alpha,ic,rmse,ic_rank,rmse_rank,final_rank
0,0.0001,1.276460,0.016861,9.537241,13.035172,11.286207
1,0.0005,1.276628,0.016861,9.505517,12.641379,11.073448
2,0.0010,1.276617,0.016861,9.504483,12.246207,10.875345
3,0.0050,1.276353,0.016861,9.550690,11.849655,10.700172
4,0.0100,1.276891,0.016860,9.513103,11.459310,10.486207
5,0.0500,1.278698,0.016858,9.554483,11.038621,10.296552
6,0.1000,1.282223,0.016855,9.456207,10.673103,10.064655
7,0.5000,1.296460,0.016840,9.390000,10.240000,9.815000
8,1.0000,1.299767,0.016829,9.508276,9.834483,9.671379
9,5.0000,1.285372,0.016794,9.516552,9.441379,9.478966


### Saving Ridge Models Comparison

In [28]:
model_comparison.to_csv(
    os.path.join(data_store,"ridge_model_comparison.csv"),
    index=False
)

## Model Rankings

In [29]:
model_rankings=(model_comparison
                .reset_index()
                .set_index("ridge_alpha")
                .sort_index()
                .final_rank
                .rank(ascending=True)
                .to_frame("Model_Rank")
                .reset_index()
               )

In [30]:
model_rankings.index=[f"Model_{i+1}" for i in range(len(ridge_alphas))]

In [31]:
model_rankings.index.name="Models"

In [32]:
model_rankings

,ridge_alpha,Model_Rank
Models,,
Model_1,0.0001,18.0
Model_2,0.0005,17.0
Model_3,0.0010,16.0
Model_4,0.0050,15.0
Model_5,0.0100,14.0
Model_6,0.0500,13.0
Model_7,0.1000,12.0
Model_8,0.5000,11.0
Model_9,1.0000,10.0


In [33]:
top5_models=model_rankings[model_rankings.Model_Rank<=5].sort_values("Model_Rank")

In [34]:
top5_models

,ridge_alpha,Model_Rank
Models,,
Model_18,50000.0,1.0
Model_17,10000.0,2.0
Model_16,5000.0,3.0
Model_15,1000.0,4.0
Model_14,500.0,5.0


## Saving Top-5 Ridge Models

In [35]:
top5_models.to_csv(
    os.path.join(data_store,"top5_ridge_models.csv"),
    index=True
)

# Training Top-5 Ridge Models

## Model-18 (ridge_alpha=50000)

### Training

In [36]:
alpha=top5_models.loc["Model_18"].ridge_alpha
ridge_model18=Ridge(
    alpha=alpha,
    fit_intercept=False,
    random_state=42
)

In [37]:
model_trainer=TrainModel(
    model=ridge_model18,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target="forward_returns_1d"
)

In [38]:
trained_ridge18,ridge18_preds,ridge18_scores=model_trainer.train(X=train_X,Y=train_Y)

### Saving Trained Model

In [40]:
joblib.dump(
    trained_ridge18,
    os.path.join(data_store,"ridge_18.pkl")
)

['/home/spiralmonster/Projects/AlphaLine/Prepared_Data_Store/ridge_18.pkl']

### Saving Model Predictions

In [43]:
ridge18_preds.sort_index(inplace=True)

In [44]:
ridge18_preds

yreal    ypreds
date       ticker                    
2010-05-20 A       0.009665 -0.004077
           ACGL   -0.002763 -0.000284
           ACN     0.004742 -0.002589
           ADI     0.018492 -0.003398
           ADM     0.001182 -0.000373
...                     ...       ...
2016-02-23 XEL     0.002009  0.001138
           XOM     0.003570 -0.001983
           YUM    -0.000420  0.002340
           ZBH     0.008444 -0.000627
           ZBRA    0.017136 -0.001002

[542300 rows x 2 columns]

In [45]:
ridge18_preds.to_csv(
    os.path.join(data_store,"ridge18_preds.csv"),
    index=True
)

### Saving Model Scores

In [47]:
ridge18_scores.sort_index(inplace=True)
ridge18_scores

,ic,rmse
date,,
2010-05-20,-29.106349,0.025418
2010-05-21,26.025158,0.016375
2010-05-24,-27.316409,0.014268
2010-05-25,8.729672,0.013683
2010-05-26,-36.529038,0.043337
...,...,...
2016-02-17,4.586007,0.019389
2016-02-18,-3.326106,0.017950
2016-02-19,22.733243,0.021853


In [48]:
ridge18_scores.to_csv(
    os.path.join(data_store,"ridge18_scores.csv"),
    index=True
)

## Model-17 (ridge_alpha=10000)

### Training

In [49]:
alpha=top5_models.loc["Model_17"].ridge_alpha
ridge_model17=Ridge(
    alpha=alpha,
    fit_intercept=False,
    random_state=42
)

In [50]:
model_trainer=TrainModel(
    model=ridge_model17,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target="forward_returns_1d"
)

In [51]:
trained_ridge17,ridge17_preds,ridge17_scores=model_trainer.train(X=train_X,Y=train_Y)

### Saving Trained Model

In [52]:
joblib.dump(
    trained_ridge17,
    os.path.join(data_store,"ridge_17.pkl")
)

['/home/spiralmonster/Projects/AlphaLine/Prepared_Data_Store/ridge_17.pkl']

### Saving Model Predictions

In [53]:
ridge17_preds.sort_index().to_csv(
    os.path.join(data_store,"ridge17_preds.csv"),
    index=True
)

### Saving Model Scores

In [54]:
ridge17_scores.sort_index().to_csv(
    os.path.join(data_store,"ridge17_scores.csv"),
    index=True
)

## Model-16 (ridge_alpha=5000)

### Training

In [55]:
alpha=top5_models.loc["Model_16"].ridge_alpha
ridge_model16=Ridge(
    alpha=alpha,
    fit_intercept=False,
    random_state=42
)

In [56]:
model_trainer=TrainModel(
    model=ridge_model16,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target="forward_returns_1d"
)

In [57]:
trained_ridge16,ridge16_preds,ridge16_scores=model_trainer.train(X=train_X,Y=train_Y)

### Saving Trained Model

In [58]:
joblib.dump(
    trained_ridge16,
    os.path.join(data_store,"ridge_16.pkl")
)

['/home/spiralmonster/Projects/AlphaLine/Prepared_Data_Store/ridge_16.pkl']

### Saving Model Predictions

In [59]:
ridge16_preds.sort_index().to_csv(
    os.path.join(data_store,"ridge16_preds.csv"),
    index=True
)

### Saving Model Scores

In [60]:
ridge16_scores.sort_index().to_csv(
    os.path.join(data_store,"ridge16_scores.csv"),
    index=True
)

## Model-15 (ridge_alpha=1000)

### Training

In [61]:
alpha=top5_models.loc["Model_15"].ridge_alpha
ridge_model15=Ridge(
    alpha=alpha,
    fit_intercept=False,
    random_state=42
)

In [62]:
model_trainer=TrainModel(
    model=ridge_model15,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target="forward_returns_1d"
)

In [63]:
trained_ridge15,ridge15_preds,ridge15_scores=model_trainer.train(X=train_X,Y=train_Y)

### Saving Trained Model

In [64]:
joblib.dump(
    trained_ridge15,
    os.path.join(data_store,"ridge_15.pkl")
)

['/home/spiralmonster/Projects/AlphaLine/Prepared_Data_Store/ridge_15.pkl']

### Saving Model Predictions

In [65]:
ridge15_preds.sort_index().to_csv(
    os.path.join(data_store,"ridge15_preds.csv"),
    index=True
)

### Saving Model Scores

In [66]:
ridge15_scores.sort_index().to_csv(
    os.path.join(data_store,"ridge15_scores.csv"),
    index=True
)

## Model-14 (ridge_alpha=500)

### Training

In [67]:
alpha=top5_models.loc["Model_14"].ridge_alpha
ridge_model14=Ridge(
    alpha=alpha,
    fit_intercept=False,
    random_state=42
)

In [68]:
model_trainer=TrainModel(
    model=ridge_model14,
    training_period=training_period,
    test_period=test_period,
    cv_splits=n_splits,
    lookahead=lookahead,
    target="forward_returns_1d"
)

In [69]:
trained_ridge14,ridge14_preds,ridge14_scores=model_trainer.train(X=train_X,Y=train_Y)

### Saving Trained Model

In [70]:
joblib.dump(
    trained_ridge14,
    os.path.join(data_store,"ridge_14.pkl")
)

['/home/spiralmonster/Projects/AlphaLine/Prepared_Data_Store/ridge_14.pkl']

### Saving Model Predictions

In [71]:
ridge14_preds.sort_index().to_csv(
    os.path.join(data_store,"ridge14_preds.csv"),
    index=True
)

### Saving Model Scores

In [72]:
ridge14_scores.sort_index().to_csv(
    os.path.join(data_store,"ridge14_scores.csv"),
    index=True
)

# END